# 🌾 Planting Method Detection with Vertex AI + Earth Engine

This notebook:
1. Loads Sentinel-1 time series data from Earth Engine
2. Sends data to Vertex AI model for prediction
3. Visualizes predictions on an interactive map
4. Shows classification results (Dry vs Wet planting method)

## 📦 Installation & Setup

In [ ]:
!pip install -q rasterio google-cloud-aiplatform

In [ ]:
# =========================
# IMPORTS
# =========================

import os
import ee
import glob
import time
import geemap
import pprint
import sqlite3
import numpy as np
import pandas as pd
import seaborn as sns
import geopandas as gpd
import tensorflow as tf
import matplotlib.pyplot as plt

from tqdm import tqdm
from datetime import datetime, timedelta
from rasterio.features import rasterize
from rasterio.transform import from_bounds

import torch
import torch.nn as nn

import google.auth
from google.colab import auth
from google.auth import compute_engine
from google.oauth2 import service_account
from google.cloud import storage
from google.cloud import aiplatform


print("✓ Imports successful")

✓ Imports successful


In [ ]:
auth.authenticate_user()

In [ ]:
# Google Cloud Configuration

# Cloud Project
PROJECT = 'sensesaka2024'
credentials, _ = google.auth.default()

# Region
REGION = 'asia-southeast1'

# Initialize Earth Engine
# ee.Authenticate()

ee.Initialize(credentials, project=PROJECT, opt_url='https://earthengine-highvolume.googleapis.com')
print("✓ GEE initialized successfully")

✓ GEE initialized successfully


## 🤖 Vertex AI Model Setup

In [ ]:
# =========================
# CONFIGURATION
# =========================

# Vertex AI Configuration
PROJECT_ID = "sensesaka2024"
REGION_VERTEX = "us-central1"  # Different from REGION above
ENDPOINT_NAME = "planting-ee-v2-endpoint"

# Model endpoint path (update this with your actual endpoint)
ENDPOINT_PATH = 'projects/246195879822/locations/asia-southeast1/endpoints/8870615721315401728'

# Model Parameters
SEQUENCE_LENGTH = 18
PATCH_SIZE = 256
SCALE = 10

# SAR Parameters
POLARIZATION = 'VH'
INTERVAL_DAYS = 14

# Time Range
START_DATE = '2023-10-01'
END_DATE = '2024-06-30'

# Tile Configuration
# TILE_GPKG_PATH = '/content/drive/MyDrive/AI-PlantingMethod/data/boundary/tile_philippines.gpkg'  # For full run
TILE_TABLE_NAME = 'tile_philippines__grid'

# Export Configuration
OUTPUT_FOLDER = 'AI-PlantingMethod'
EXPORT_SCALE = 10
EXPORT_CRS = 'EPSG:4326'
MAX_PIXELS = 1e13
NODATA_VALUE = -9999

print("✓ Configuration loaded")
print(f"\nTime Range: {START_DATE} to {END_DATE}")

✓ Configuration loaded

Time Range: 2023-10-01 to 2024-06-30


In [ ]:
# Check GPU availability
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")

if torch.cuda.is_available():
  print(f"GPU: {torch.cuda.get_device_name(0)}")

Using device: cpu


## 🛰️ Sentinel-1 Data Processing Functions

In [ ]:
gpkg_dir = '/content/drive/MyDrive/AI-PlantingMethod/data/boundary'

bounds_list = []

for file in os.listdir(gpkg_dir):
    if file.endswith('.gpkg'):
        file_path = os.path.join(gpkg_dir, file)
        gdf = gpd.read_file(file_path)

        min_lon, min_lat, max_lon, max_lat = gdf.total_bounds
        bounds_list.append({
            "filename": file,
            "min_lon": min_lon,
            "min_lat": min_lat,
            "max_lon": max_lon,
            "max_lat": max_lat
        })

# convert to DataFrame
bounds_df = pd.DataFrame(bounds_list)
print(bounds_df)

/usr/local/lib/python3.12/dist-packages/pyogrio/geopandas.py:382: UserWarning: More than one layer found in 'tile_test.gpkg': 'tile_philippines__grid' (default), 'tile_philippines_01__grid'. Specify layer parameter to avoid this warning.
  result = read_func(


                      filename     min_lon    min_lat     max_lon    max_lat
0   tile_philippines_0.25.gpkg  117.178312   5.466135  126.678312  18.966135
1               tile_test.gpkg  121.528337  16.670141  121.928337  17.070141
2            tile_visayas.gpkg  121.828337   8.970141  125.828337  12.570141
3            tile_antique.gpkg  121.282669  10.406292  122.324601  12.208996
4             tile_iloilo.gpkg  122.012161  10.466848  123.372770  11.638852
5              tile_capiz.gpkg  122.199908  11.133469  123.098359  11.646015
6              tile_aklan.gpkg  121.843596  11.311230  122.577607  11.999384
7           tile_guimaras.gpkg  122.474695  10.386120  122.743820  10.754278
8   tile_negrosoccidental.gpkg  122.364871   9.444193  123.572899  11.044571
9     tile_negrosoriental.gpkg  122.588892   9.038368  123.335915  10.439601
10              tile_cebu.gpkg  123.296368   9.412867  124.570972  11.524289
11             tile_bohol.gpkg  123.681254   9.485586  124.645375  10.252672

In [ ]:
area = 'tile_philippines_0.25.gpkg'

min_lon = bounds_df[bounds_df['filename'] == area]['min_lon']
min_lat = bounds_df[bounds_df['filename'] == area]['min_lat']
max_lon = bounds_df[bounds_df['filename'] == area]['max_lon']
max_lat = bounds_df[bounds_df['filename'] == area]['max_lat']

min_lon = min_lon.values[0]
min_lat = min_lat.values[0]
max_lon = max_lon.values[0]
max_lat = max_lat.values[0]

# =========================
# DEFINE AOI AND TIME RANGE
# =========================

# min_lon, min_lat, max_lon, max_lat = 119.735, 12.516, 121.735, 15.516
aoi = ee.Geometry.Rectangle([min_lon, min_lat, max_lon, max_lat])

start_date = '2024-10-01'
end_date = '2025-06-30'

TILE_GPKG_PATH = f'/content/drive/MyDrive/AI-PlantingMethod/data/boundary/{area}'
print(TILE_GPKG_PATH)
print(f"\nAOI Bounds: ({min_lon}, {min_lat}) to ({max_lon}, {max_lat})")
print(f"Time Range: {start_date} to {end_date}\n")

/content/drive/MyDrive/AI-PlantingMethod/data/boundary/tile_philippines_0.25.gpkg

AOI Bounds: (117.178311786, 5.466135168000001) to (126.678311786, 18.966135168)
Time Range: 2024-10-01 to 2025-06-30



In [ ]:
def load_tiles_from_gpkg(gpkg_path):
    """
    Load tiles from GeoPackage SQLite database.

    The table name is always 'tile_philippines__grid' in your GeoPackages.

    Returns:
        List of tile dictionaries with geometry information
    """
    conn = sqlite3.connect(gpkg_path)
    cursor = conn.cursor()

    # The table is always named 'tile_philippines__grid'
    query = """
    SELECT fid, id, left, top, right, bottom, row_index, col_index
    FROM grid
    ORDER BY row_index, col_index
    """

    cursor.execute(query)
    rows = cursor.fetchall()

    tiles = []
    for row in rows:
        tile = {
            'fid': row[0],
            'id': row[1],
            'left': row[2],
            'top': row[3],
            'right': row[4],
            'bottom': row[5],
            'row_index': row[6],
            'col_index': row[7],
            'geometry': ee.Geometry.Rectangle(
                [row[2], row[5], row[4], row[3]],  # [left, bottom, right, top]
                proj='EPSG:4326',
                geodesic=False
            )
        }
        tiles.append(tile)

    conn.close()
    return tiles

print("✓ Tile loading function defined")

✓ Tile loading function defined


In [ ]:
# Load tiles
tiles = load_tiles_from_gpkg(TILE_GPKG_PATH)

print(f"✓ Loaded {len(tiles)} tiles from GeoPackage")
print(f"\nFirst 5 tiles:")
for i, tile in enumerate(tiles[:5]):
    print(f"  [{i+1}] Tile {tile['id']}: Row {tile['row_index']}, Col {tile['col_index']}")
    print(f"      Bounds: ({tile['left']:.4f}, {tile['bottom']:.4f}) to ({tile['right']:.4f}, {tile['top']:.4f})")

✓ Loaded 680 tiles from GeoPackage

First 5 tiles:
  [1] Tile 933: Row 4, Col 16
      Bounds: (121.1783, 18.7161) to (121.4283, 18.9661)
  [2] Tile 991: Row 4, Col 17
      Bounds: (121.4283, 18.7161) to (121.6783, 18.9661)
  [3] Tile 760: Row 5, Col 13
      Bounds: (120.4283, 18.4661) to (120.6783, 18.7161)
  [4] Tile 818: Row 5, Col 14
      Bounds: (120.6783, 18.4661) to (120.9283, 18.7161)
  [5] Tile 876: Row 5, Col 15
      Bounds: (120.9283, 18.4661) to (121.1783, 18.7161)


In [ ]:
polarization = 'VH'

def smoothing(image):

  return image.focalMode(**{'radius':1, 'kernelType':'circle', 'units':'pixels', 'iterations':10})

def mask_edge(image):
    """Mask out edges with very low backscatter"""
    edge = image.lt(-30.0)
    masked_image = image.mask().And(edge.Not())
    return image.updateMask(masked_image)


def refined_lee_filter(image, polarization='VH', kernel_size=3):
    """
    Apply Refined Lee filter to reduce speckle noise
    FIXED: Properly handles array operations and returns regular image

    Parameters:
        image: ee.Image with polarization band
        polarization: str, polarization type ('VV' or 'VH')
        kernel_size: int, kernel size (default 3)

    Returns:
        ee.Image with filtered band
    """
    # Select the polarization band
    img_band = image.select(polarization)

    # Convert from dB to linear scale
    img_linear = ee.Image(10).pow(img_band.divide(10))

    # Define kernel
    kernel = ee.Kernel.square(kernel_size / 2, 'pixels', False)

    # Calculate local mean
    mean = img_linear.reduceNeighborhood(
        reducer=ee.Reducer.mean(),
        kernel=kernel
    )

    # Calculate local variance
    variance = img_linear.reduceNeighborhood(
        reducer=ee.Reducer.variance(),
        kernel=kernel
    )

    # Estimate noise variance
    # Use a simpler approach that doesn't create array issues
    sample_weights = variance.divide(mean.pow(2))

    # Create a larger kernel for noise estimation
    noise_kernel = ee.Kernel.square(kernel_size * 2, 'pixels', False)

    # Get noise variance estimate (using minimum in neighborhood)
    noise_variance = sample_weights.reduceNeighborhood(
        reducer=ee.Reducer.min(),
        kernel=noise_kernel
    )

    # Refined Lee formula
    # varX = (variance - mean^2 * noise_variance) / (noise_variance + 1)
    varX = variance.subtract(
        mean.pow(2).multiply(noise_variance)
    ).divide(
        noise_variance.add(1.0)
    )

    # Weighting factor
    b = varX.divide(variance)

    # Apply filter
    filtered = mean.add(
        b.multiply(img_linear.subtract(mean))
    )

    # Convert back to dB
    filtered_db = filtered.log10().multiply(10)

    # Get date for naming
    date_str = ee.Date(image.get('system:time_start')).format('YYYY-MM-dd')
    custom_name = date_str.cat(f'_S1{polarization}')

    # Return as regular image (not array)
    return (filtered_db
            .rename([polarization])
            .set('name', custom_name)
            .set('key', custom_name)
            .copyProperties(image, ['system:time_start', 'system:id']))

# Fix empty mosaics to avoid 0-band images
def fix_empty(img, band_name):
    img = ee.Image(img)  # if img is NULL → returns a NULL placeholder
    return ee.Image(
        ee.Algorithms.If(
            img,
            ee.Algorithms.If(
                img.bandNames().size().eq(0),
                ee.Image(0).rename(band_name).updateMask(ee.Image(1)),
                img
            ),
            ee.Image(0).rename(band_name).updateMask(ee.Image(1))
        )
    )

def temporal_linear_interpolation(ic):

    def interp(img):
        img = ee.Image(img)
        t = img.date().millis()

        # Get previous image
        prev = ic.filterDate(
            ee.Date(t).advance(-500, 'day'),
            ee.Date(t)
        ).sort('system:time_start', False).first()

        # Get next image
        next_ = ic.filterDate(
            ee.Date(t),
            ee.Date(t).advance(500, 'day')
        ).sort('system:time_start').first()

        # Ensure prev/next aren't empty
        prev = fix_empty(prev, polarization)
        next_ = fix_empty(next_, polarization)
        img  = fix_empty(img, polarization)

        prev = ee.Image(prev)
        next_ = ee.Image(next_)

        prev_t = ee.Number(prev.get('system:time_start'))
        next_t = ee.Number(next_.get('system:time_start'))

        # Avoid divide-by-zero in rare cases
        ratio = ee.Number(t).subtract(prev_t).divide(
            next_t.subtract(prev_t).max(1)
        )

        # Linear interpolation: prev + ratio*(next - prev)
        interpolated = prev.add(
            next_.subtract(prev).multiply(ratio)
        )

        # Fill missing pixels in original with interpolated values
        filled = img.unmask(interpolated)

        return filled.copyProperties(img).set('system:time_start', t)

    return ic.map(interp)

In [ ]:
collectionS1 = (ee.ImageCollection('COPERNICUS/S1_GRD')
                .filter(ee.Filter.eq('instrumentMode', 'IW'))
                .filterDate(start_date, end_date)
                .filter(ee.Filter.listContains('transmitterReceiverPolarisation', polarization))
                .filterBounds(aoi)
                .map(mask_edge))

# Filter by descending orbit properties only
collectionS1_desc = collectionS1.filter(ee.Filter.eq('orbitProperties_pass', 'DESCENDING'))

# Select VH band and apply a refined Lee filter
collectionS1_desc_VH = collectionS1_desc.select(['VH']).map(refined_lee_filter)

# Function to sample images at biweekly intervals (14 days)
def sample_images_at_intervals(collection, start_date, end_date, interval_days):
    date_list = ee.List.sequence(ee.Date(start_date).millis(), ee.Date(end_date).millis(), interval_days * 24 * 60 * 60 * 1000)

    def filter_and_get_image(millis):
        filtered = collection.filterDate(ee.Date(millis), ee.Date(millis).advance(interval_days, 'day')).mosaic()
        renamed_image = filtered.rename(polarization)
        image_date = ee.Date(millis).format('YYYY-MM-dd')
        image_name = ee.String('S1_Mosaic_').cat(image_date)
        return renamed_image.set('system:time_start', millis, 'system:id', image_name)

    sampled_images = date_list.map(filter_and_get_image)
    return ee.ImageCollection.fromImages(sampled_images)

# Sample images at BIWEEKLY intervals (changed from 1 to 14)
biweekly_images_desc_mosaicked = sample_images_at_intervals(collectionS1_desc_VH, start_date, end_date, 14)

# Sort the biweekly mosaicked images by date
sorted_combined_images = biweekly_images_desc_mosaicked.sort('system:time_start')

sorted_combined_images

In [ ]:

def create_linear_interpolated_timeseries(aoi, start_date, end_date, polarization='VH',
                                         interval_days=14, expected_count=18):
    """
    Linear interpolation with better missing value handling.
    Interpolates temporally between available images and fills gaps.
    """

    # Collect S1 data
    collectionS1 = (ee.ImageCollection('COPERNICUS/S1_GRD')
                    .filter(ee.Filter.eq('instrumentMode', 'IW'))
                    .filterDate(start_date, end_date)
                    .filter(ee.Filter.listContains('transmitterReceiverPolarisation', polarization))
                    .filterBounds(aoi)
                    .map(mask_edge))

    collectionS1_desc = collectionS1.filter(ee.Filter.eq('orbitProperties_pass', 'DESCENDING'))
    collectionS1_desc_pol = collectionS1_desc.select([polarization]).map(refined_lee_filter)

    # Generate target dates
    date_list = ee.List.sequence(
        ee.Date(start_date).millis(),
        ee.Date(end_date).millis(),
        interval_days * 24 * 60 * 60 * 1000
    ).slice(0, expected_count)

    # For each target date, find closest images and interpolate
    def interpolate_for_date(millis):
        target_date = ee.Date(millis)

        # Extended search window for better interpolation
        search_window = interval_days * 2  # Look further for interpolation candidates
        window_start = target_date.advance(-search_window, 'day')
        window_end = target_date.advance(search_window, 'day')

        nearby = collectionS1_desc_pol.filterDate(window_start, window_end)

        # Check for exact match in smaller window
        exact_window = interval_days / 2
        exact_match = collectionS1_desc_pol.filterDate(
            target_date.advance(-exact_window, 'day'),
            target_date.advance(exact_window, 'day')
        )

        has_exact = exact_match.size().gt(0)

        # If exact match exists, use it
        def use_exact():
            return exact_match.mosaic()

        # Otherwise, interpolate from nearby images
        def interpolate_nearby():
            # Get images before and after target date
            before = nearby.filterDate(window_start, target_date).sort('system:time_start', False)
            after = nearby.filterDate(target_date, window_end).sort('system:time_start')

            has_before = before.size().gt(0)
            has_after = after.size().gt(0)

            # Linear interpolation between closest before and after images
            def linear_interp():
                img_before = ee.Image(before.first())
                img_after = ee.Image(after.first())

                date_before = ee.Date(img_before.get('system:time_start'))
                date_after = ee.Date(img_after.get('system:time_start'))

                # Calculate weights based on temporal distance
                total_diff = date_after.difference(date_before, 'day')
                weight_after = target_date.difference(date_before, 'day').divide(total_diff)
                weight_before = ee.Number(1).subtract(weight_after)

                # Weighted average for smooth interpolation
                interpolated = img_before.multiply(weight_before).add(
                    img_after.multiply(weight_after)
                )

                return interpolated

            # Fallback: use nearest available image
            def use_before():
                return before.first()

            def use_after():
                return after.first()

            # Last resort: use median of all available data
            def use_fallback():
                all_images = collectionS1_desc_pol
                return ee.Algorithms.If(
                    all_images.size().gt(0),
                    all_images.median(),
                    ee.Image.constant(0).rename(polarization)  # Fill with zeros if no data
                )

            # Decision tree for interpolation strategy
            return ee.Image(ee.Algorithms.If(
                has_before.And(has_after),
                linear_interp(),
                ee.Algorithms.If(
                    has_before,
                    use_before(),
                    ee.Algorithms.If(
                        has_after,
                        use_after(),
                        use_fallback()
                    )
                )
            ))

        result = ee.Image(ee.Algorithms.If(has_exact, use_exact(), interpolate_nearby()))
        return result.set('system:time_start', millis).set('interpolated', ee.Number(has_exact).Not())

    # Create interpolated collection
    interpolated_images = date_list.map(interpolate_for_date)
    interpolated_collection = ee.ImageCollection.fromImages(interpolated_images)

    # Stack into multi-band image
    band_list = ee.List.sequence(0, expected_count - 1)

    def rename_band(idx):
        idx = ee.Number(idx)
        img = ee.Image(interpolated_images.get(idx))
        band_name = idx.format('%d').cat('_').cat(polarization)
        return img.select([polarization]).rename(band_name)

    renamed_images = band_list.map(rename_band)

    # Combine all bands
    stacked = ee.ImageCollection.fromImages(renamed_images).toBands()

    # Remove the collection prefix from band names
    old_names = stacked.bandNames()

    def create_band_name(i):
        return ee.Number(i).format('%d').cat('_').cat(polarization)

    new_names = band_list.map(create_band_name)

    final_stack = stacked.rename(new_names)

    # Optional: Apply gap-filling for any remaining missing pixels
    final_stack = fill_remaining_gaps(final_stack, expected_count, polarization)

    return final_stack


def fill_remaining_gaps(stacked_image, band_count, polarization):
    """
    Fill any remaining gaps using linear interpolation across the time series.
    This handles pixel-level missing values within bands.
    """
    def create_band_name(i):
        return ee.Number(i).format('%d').cat('_').cat(polarization)

    band_names = ee.List.sequence(0, band_count - 1).map(create_band_name)

    def interpolate_band(band_idx):
        band_idx = ee.Number(band_idx)
        band_name = band_idx.format('%d').cat('_').cat(polarization)
        current_band = stacked_image.select([band_name])

        # Find previous and next valid bands for interpolation
        prev_idx = band_idx.subtract(1)
        next_idx = band_idx.add(1)

        has_prev = prev_idx.gte(0)
        has_next = next_idx.lt(band_count)

        def interp_both():
            prev_name = prev_idx.format('%d').cat('_').cat(polarization)
            next_name = next_idx.format('%d').cat('_').cat(polarization)
            prev_band = stacked_image.select([prev_name])
            next_band = stacked_image.select([next_name])
            return prev_band.add(next_band).divide(2)

        def use_prev():
            prev_name = prev_idx.format('%d').cat('_').cat(polarization)
            return stacked_image.select([prev_name])

        def use_next():
            next_name = next_idx.format('%d').cat('_').cat(polarization)
            return stacked_image.select([next_name])

        interpolated = ee.Image(ee.Algorithms.If(
            has_prev.And(has_next),
            interp_both(),
            ee.Algorithms.If(has_prev, use_prev(),
                ee.Algorithms.If(has_next, use_next(), current_band))
        ))

        # Use interpolated values only where original is masked
        return current_band.unmask(interpolated).rename([band_name])

    # Apply interpolation to all bands
    band_indices = ee.List.sequence(0, band_count - 1)
    interpolated_bands = band_indices.map(interpolate_band)

    return ee.ImageCollection.fromImages(interpolated_bands).toBands().rename(band_names)

In [ ]:
composite_interpolated = create_linear_interpolated_timeseries(
    aoi, start_date, end_date,
    polarization=polarization,
    interval_days=14,
    expected_count=18
)

composite_interpolated

In [ ]:
# Load ESA WorldCover for 2020 and select the cropland class (class 40)
# esa_worldcover = ee.Image("ESA/WorldCover/v100/2020").select('Map')

# Create a cropland mask (1 for cropland, 0 for non-cropland)
# cropland_mask = esa_worldcover.eq(40)

temporary_crops = ee.ImageCollection("ESA/WorldCereal/2021/MODELS/v100") \
  .filter(ee.Filter.eq('product', 'temporarycrops')) \
  .filter(ee.Filter.eq('season', 'tc-annual'))

# Get the classification band (0 or 100)
temporary_crops_class = temporary_crops.mosaic().select('classification')

# Create mask where temporary crops exist
# Note: WorldCereal uses 0=no crop, 100=crop
crop_mask = temporary_crops_class.eq(100)

composite_mask = composite_interpolated.updateMask(crop_mask)

composite_mask

In [ ]:
def visualize_timeseries(time_series, aoi, polarization='VH'):
    """
    Create map visualization

    Args:
        time_series: ee.ImageCollection
        aoi: Area of interest
        polarization: 'VV' or 'VH'

    Returns:
        geemap.Map
    """
    import geemap

    Map = geemap.Map()
    Map.centerObject(aoi, 12)

    # Add AOI
    Map.addLayer(ee.Image().paint(aoi, 0, 2), {'palette': 'yellow'}, 'AOI')

    # Visualization parameters
    vis = {
        'bands': ['0_VH', '5_VH', '10_VH'],
        'min': -30,
        'max': -5
    }


    start_date = time_series.get('start_date').getInfo()
    end_date = time_series.get('end_date').getInfo()


    # Create descriptive layer name
    layer_name = f'{start_date} to {end_date} - {polarization}'
    Map.addLayer(time_series, vis, layer_name, False)

    # Add latest image as visible by default
    latest_start = time_series.get('start_date').getInfo()
    latest_end = time_series.get('end_date').getInfo()
    latest_period = time_series.get('period').getInfo()

    latest_layer_name = f'★ Latest ({latest_period}): {latest_start} to {latest_end}'
    Map.addLayer(time_series, vis, latest_layer_name, True)


    return Map

  # Date range for biweekly composites

print(f"Creating visualization for first tile...")
example_tile = tiles[0]
aoi = ee.Geometry.Rectangle([113.6804, 3.1114, 129.9438, 22.2536])
Map = visualize_timeseries(composite_mask, aoi, 'VH')
Map

Creating visualization for first tile...


Map(center=[12.679436406891591, 121.81210000000002], controls=(WidgetControl(options=['position', 'transparent…

In [ ]:
# ============================================
# APPLY MAJORITY FILTER
# ============================================

def majority_filter(image, radius=1):
    """
    Apply majority filter to reduce noise.

    Args:
        image: Binary classification image (0 or 1)
        radius: Kernel radius in pixels (default=1, creates 3x3 kernel)

    Returns:
        Filtered image
    """
    # Create kernel
    kernel = ee.Kernel.square(radius=radius, units='pixels')

    # Apply mode (majority) filter
    # This replaces each pixel with the most common value in the neighborhood
    filtered = image.reduceNeighborhood(
        reducer=ee.Reducer.mode(),
        kernel=kernel
    )

    return filtered.toByte()


def process_and_export_tile(tile, vertex_model, buffer_degrees=0.001):
    """
    Process a single tile with buffer in DEGREES (for EPSG:4326)

    Args:
        tile: Dictionary with tile info (geometry, id, row_index, col_index)
        vertex_model: Vertex AI model endpoint
        buffer_degrees: Buffer distance in DEGREES

    Returns:
        ee.batch.Task or None if error
    """

    OUTPUT_BUCKET = 'leads-agri'
    OUTPUT_PATH = 'tmp/inference'

    try:
        tile_id = tile['id']
        row = tile['row_index']
        col = tile['col_index']

        print(f"  Tile {tile_id} (Row {row}, Col {col})")

        # Get original geometry
        original_geometry = tile['geometry']

        # Create buffered geometry for processing (in degrees!)
        buffered_geometry = original_geometry.buffer(buffer_degrees)

        print(f"    Buffer: {buffer_degrees} degrees (≈{buffer_degrees*111:.0f}m at equator)")

        # Prepare input with BUFFERED geometry
        expected_bands = [f"{i}_{POLARIZATION}" for i in range(SEQUENCE_LENGTH)]
        composite_ordered = composite_mask.clip(buffered_geometry).select(expected_bands).updateMask(crop_mask)
        input_array = composite_ordered.float()

        # Run prediction on buffered area
        print("    Predicting (buffered area)...")
        predictions_ = vertex_model.predictImage(input_array).select(['predictions']).arrayGet([0])

        # CRITICAL: Clip back to ORIGINAL geometry before export
        output_bands = predictions_.bandNames().getInfo()
        print(f"  Output bands: {output_bands}")

        predictions = predictions_.gt(0.3).toByte()
        predictions_clipped = predictions.clip(buffered_geometry).updateMask(crop_mask)
        result_filtered = majority_filter(predictions_clipped, radius=3)

        # Export with proper NoData handling
        print("    Exporting (original bounds)...")
        filename = f"tile_{tile_id}_r{row}_c{col}"
        file_prefix = f'{OUTPUT_PATH}/{filename}'

        # Unmask to set NoData value for masked pixels
        prediction_export = result_filtered.unmask(NODATA_VALUE, False)

        task = ee.batch.Export.image.toCloudStorage(
            image=prediction_export.toFloat(),
            description=filename,
            bucket=OUTPUT_BUCKET,
            fileNamePrefix=file_prefix,
            region=buffered_geometry,  # Export only original bounds
            scale=EXPORT_SCALE,
            crs=EXPORT_CRS,
            maxPixels=MAX_PIXELS,
            fileFormat='GeoTIFF',
            formatOptions={
                'cloudOptimized': True,
                'noData': NODATA_VALUE
            }
        )

        task.start()
        print(f"    ✓ Task started: {task.id}")
        return task

    except Exception as e:
        print(f"    ✗ Error: {str(e)}")
        import traceback
        traceback.print_exc()
        return None


print("✓ Processing function defined")

✓ Processing function defined


In [ ]:
# ============================================
# CONNECT TO MODEL
# ============================================

print("\n" + "="*70)
print("CONNECTING TO VERTEX AI MODEL")
print("="*70)

# For IMAGE data with TensorFlow model
vertex_model = ee.Model.fromVertexAi(
    endpoint='projects/246195879822/locations/asia-southeast1/endpoints/3568471585016774656',
    inputTileSize=[16, 16],  # Process in 64x64 tiles
    proj=ee.Projection('EPSG:4326').atScale(10),
    fixInputProj=True,
    outputBands={
        'predictions': {
            'type': ee.PixelType.float(),
            'dimensions': 1
        }
    }
)

print("✓ Model connected")

# ============================================
# RUN PREDICTION
# ============================================

print("\n" + "="*70)
print("RUNNING PREDICTION")
print("="*70)

tasks = []

for i, tile in enumerate(tiles):
    print(f"\n[{i+1}/{len(tiles)}]")
    task = process_and_export_tile(tile, vertex_model)
    if task:
        tasks.append(task)

    # Small delay to avoid overwhelming the system
    time.sleep(2)

print(f"\n{'='*70}")
print(f"Started {len(tasks)}/{len(tiles)} export tasks")
print(f"{'='*70}")

Streaming output truncated to the last 5000 lines.

[54/680]
  Tile 708 (Row 11, Col 12)
    Buffer: 0.001 degrees (≈0m at equator)
    Predicting (buffered area)...
  Output bands: ['predictions']
    Exporting (original bounds)...
    ✓ Task started: TE4SVOJT4JILWZWEXIAVYBE3

[55/680]
  Tile 766 (Row 11, Col 13)
    Buffer: 0.001 degrees (≈0m at equator)
    Predicting (buffered area)...
  Output bands: ['predictions']
    Exporting (original bounds)...
    ✓ Task started: HHWIYY5PBFQV32MABRDC5N6W

[56/680]
  Tile 824 (Row 11, Col 14)
    Buffer: 0.001 degrees (≈0m at equator)
    Predicting (buffered area)...
  Output bands: ['predictions']
    Exporting (original bounds)...
    ✓ Task started: EUM3TZY36HA7J7F2QH4XJWLP

[57/680]
  Tile 882 (Row 11, Col 15)
    Buffer: 0.001 degrees (≈0m at equator)
    Predicting (buffered area)...
  Output bands: ['predictions']
    Exporting (original bounds)...
    ✓ Task started: ZLGBGOCVCO63QNF277W35ALQ

[58/680]
  Tile 940 (Row 11, Col 16)
   


[679/680]
  Tile 1914 (Row 57, Col 32)
    Buffer: 0.001 degrees (≈0m at equator)
    Predicting (buffered area)...
  Output bands: ['predictions']
    Exporting (original bounds)...
    ✓ Task started: BTYZESCZY5L7Y53GHHUVR7QH

[680/680]
  Tile 1972 (Row 57, Col 33)
    Buffer: 0.001 degrees (≈0m at equator)
    Predicting (buffered area)...
  Output bands: ['predictions']
    Exporting (original bounds)...
    ✓ Task started: EPGLM6VGM7AMMAZVRHBP4LYC

Started 680/680 export tasks


In [ ]:
# Monitor tasks
print("Monitoring export tasks...\n")
print("Check detailed status at: https://code.earthengine.google.com/tasks\n")

while True:
    statuses = {}
    for task in tasks:
        state = task.status()['state']
        statuses[state] = statuses.get(state, 0) + 1

    timestamp = datetime.now().strftime('%Y-%m-%d %H:%M:%S')
    print(f"[{timestamp}]", end=" ")
    for state, count in sorted(statuses.items()):
        print(f"{state}: {count}", end="  ")
    print()

    # Check if all done
    if all(task.status()['state'] in ['COMPLETED', 'FAILED', 'CANCELLED'] for task in tasks):
        print("\n✓ All tasks finished!")
        break

    time.sleep(60)  # Check every minute

Monitoring export tasks...

Check detailed status at: https://code.earthengine.google.com/tasks

[2026-02-18 13:36:35] READY: 2  RUNNING: 2  
[2026-02-18 13:37:37] READY: 2  RUNNING: 2  
[2026-02-18 13:38:38] READY: 2  RUNNING: 2  
[2026-02-18 13:39:39] READY: 2  RUNNING: 2  
[2026-02-18 13:40:40] READY: 2  RUNNING: 2  
[2026-02-18 13:41:41] READY: 2  RUNNING: 2  
[2026-02-18 13:42:42] READY: 2  RUNNING: 2  
[2026-02-18 13:43:44] READY: 2  RUNNING: 2  
[2026-02-18 13:44:46] COMPLETED: 1  READY: 1  RUNNING: 2  
[2026-02-18 13:45:47] COMPLETED: 1  READY: 1  RUNNING: 2  
[2026-02-18 13:46:49] COMPLETED: 2  RUNNING: 2  
[2026-02-18 13:47:51] COMPLETED: 2  RUNNING: 2  
[2026-02-18 13:48:53] COMPLETED: 2  RUNNING: 2  
[2026-02-18 13:49:55] COMPLETED: 2  RUNNING: 2  
[2026-02-18 13:50:57] COMPLETED: 2  RUNNING: 2  
[2026-02-18 13:51:59] COMPLETED: 2  RUNNING: 2  
[2026-02-18 13:53:01] COMPLETED: 2  RUNNING: 2  
[2026-02-18 13:54:02] COMPLETED: 3  RUNNING: 1  
[2026-02-18 13:55:04] COMPLETED: 

In [ ]:
# Final summary
print("\n" + "="*70)
print("FINAL SUMMARY")
print("="*70)

completed = sum(1 for t in tasks if t.status()['state'] == 'COMPLETED')
failed = sum(1 for t in tasks if t.status()['state'] == 'FAILED')
cancelled = sum(1 for t in tasks if t.status()['state'] == 'CANCELLED')

print(f"✓ Completed: {completed}/{len(tasks)}")
print(f"✗ Failed: {failed}/{len(tasks)}")
print(f"⊘ Cancelled: {cancelled}/{len(tasks)}")

if failed > 0:
    print("\nFailed tasks:")
    for task in tasks:
        if task.status()['state'] == 'FAILED':
            status = task.status()
            print(f"  - {status['description']}")
            if 'error_message' in status:
                print(f"    Error: {status['error_message']}")

print(f"\nOutputs saved to Google Drive folder: {OUTPUT_FOLDER}")

In [ ]:
tile = tiles[0]
bounds = tile['geometry']

In [ ]:
input_properties = [f"{i}_VH" for i in range(18)]
output_properties = [f"arg_{i}_VH" for i in range(18)]

# ============================================
# PREPARE DATA
# ============================================

print("\n" + "="*70)
print("PREPARING INPUT DATA")
print("="*70)

# Your composite
# composite_mask = ...
# bounds = ...

# Select bands
composite_ordered = composite_mask.clip(bounds).select(input_properties).updateMask(crop_mask)

# Unmask (replace masked pixels with -30)
input_image = composite_ordered.float()

print("✓ Input image prepared")
print(f"  Bands: {input_properties}")

# ============================================
# CONNECT TO MODEL
# ============================================

print("\n" + "="*70)
print("CONNECTING TO VERTEX AI MODEL")
print("="*70)

# For IMAGE data with TensorFlow model
vertex_model = ee.Model.fromVertexAi(
    endpoint='projects/246195879822/locations/asia-southeast1/endpoints/3568471585016774656',
    inputTileSize=[16, 16],  # Process in 64x64 tiles
    proj=ee.Projection('EPSG:4326').atScale(10),
    fixInputProj=True,
    outputBands={
        'predictions': {
            'type': ee.PixelType.float(),
            'dimensions': 1
        }
    }
)

print("✓ Model connected")

# ============================================
# RUN PREDICTION
# ============================================

print("\n" + "="*70)
print("RUNNING PREDICTION")
print("="*70)

# Use predictImage for raster data
predictions = vertex_model.predictImage(input_image).select(['predictions']).arrayGet([0])

print("✓ Prediction completed")

# Check output bands
output_bands = predictions.bandNames().getInfo()
print(f"  Output bands: {output_bands}")

# ============================================
# EXTRACT CLASSIFICATION
# ============================================

print("\n" + "="*70)
print("EXTRACTING CLASSIFICATION")
print("="*70)

#planting_class = predictions.arrayArgmax().arrayGet([0])
#result = planting_class.clip(bounds).updateMask(crop_mask)
# Convert to classes using threshold 0.5
# > 0.5 = Class 1 (Transplanted)
# <= 0.5 = Class 0 (Direct-Seeded)
planting_class = predictions.gt(0.3).toByte()

result = planting_class.clip(bounds).updateMask(crop_mask)
result



PREPARING INPUT DATA
✓ Input image prepared
  Bands: ['0_VH', '1_VH', '2_VH', '3_VH', '4_VH', '5_VH', '6_VH', '7_VH', '8_VH', '9_VH', '10_VH', '11_VH', '12_VH', '13_VH', '14_VH', '15_VH', '16_VH', '17_VH']

CONNECTING TO VERTEX AI MODEL
✓ Model connected

RUNNING PREDICTION
✓ Prediction completed
  Output bands: ['predictions']

EXTRACTING CLASSIFICATION


In [ ]:
# ============================================
# VISUALIZE
# ============================================

# ============================================
# APPLY MAJORITY FILTER
# ============================================

print("\n" + "="*70)
print("APPLYING MAJORITY FILTER")
print("="*70)

def majority_filter(image, radius=1):
    """
    Apply majority filter to reduce noise.

    Args:
        image: Binary classification image (0 or 1)
        radius: Kernel radius in pixels (default=1, creates 3x3 kernel)

    Returns:
        Filtered image
    """
    # Create kernel
    kernel = ee.Kernel.square(radius=radius, units='pixels')

    # Apply mode (majority) filter
    # This replaces each pixel with the most common value in the neighborhood
    filtered = image.reduceNeighborhood(
        reducer=ee.Reducer.mode(),
        kernel=kernel
    )

    return filtered.toByte()

result_filtered = majority_filter(result, radius=3)

# Apply filter with different radius options
print("Applying majority filter...")

print("\n" + "="*70)
print("VISUALIZING")
print("="*70)

Map = geemap.Map()
Map.centerObject(bounds, 12)

# Add classification
Map.addLayer(
    result_filtered,
    {
        'min': 0,
        'max': 1,
        'palette': ['red', 'orange']
    },
    'Planting Method',
    True,
    0.8
)

# Add boundary
Map.addLayer(bounds, {'color': 'yellow'}, 'Boundary', True, 0.5)

# Add original composite
Map.addLayer(
    composite_ordered.select([0, 1, 2]),
    {'min': -25, 'max': 0},
    'SAR Composite',
    False
)

print("✓ Map ready")

Map


APPLYING MAJORITY FILTER
Applying majority filter...

VISUALIZING


✓ Map ready


Map(center=[16.970131206997802, 121.62833709999909], controls=(WidgetControl(options=['position', 'transparent…

In [ ]:
task = ee.batch.Export.image.toDrive(
            image=predictions.clip(bounds),
            description='planting-method',
            folder=OUTPUT_FOLDER,
            fileNamePrefix='_drySeason',
            region=bounds,
            scale=EXPORT_SCALE,
            crs=EXPORT_CRS,
            maxPixels=MAX_PIXELS,
            fileFormat='GeoTIFF',
            formatOptions={
                'cloudOptimized': True,
                'noData': NODATA_VALUE
            })

task.start()